# EDA — Airline Passenger Satisfaction
Fase 2 del proyecto. Análisis exploratorio del dataset elegido en la Fase 1.

## 1. Carga de datos

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency

sns.set_style('whitegrid')

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
df_train = pd.read_csv(os.path.join(path, "train.csv"))
df_test = pd.read_csv(os.path.join(path, "test.csv"))
df = pd.concat([df_train, df_test], ignore_index=True)

print(df.shape)
df.head()

## 2. Limpieza inicial

Antes de empezar el análisis revisamos la estructura del dataframe para detectar cosas raras.

In [ ]:
print(df.info())
print("\nColumnas:", list(df.columns))

Aparece una columna `Unnamed: 0` que no está en el diccionario de variables. La revisamos antes de decidir qué hacer:

In [ ]:
print(df['Unnamed: 0'].head(10))
print(df_train['Unnamed: 0'].equals(df_train.index.to_series()))
print(df_test['Unnamed: 0'].equals(df_test.index.to_series()))

Es el índice original de cada archivo (train y test), y al unirlos ya no significa nada. La quitamos.

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
print(df.shape)

## 3. Valores faltantes

In [ ]:
faltantes = df.isnull().sum()
faltantes_pct = (faltantes / len(df) * 100).round(2)
resumen_faltantes = pd.DataFrame({'faltantes': faltantes, 'porcentaje': faltantes_pct})
print(resumen_faltantes[resumen_faltantes['faltantes'] > 0])

Solo `Arrival Delay in Minutes` tiene datos faltantes: 393 (0.3% del total). Antes de imputar, comparamos media y mediana para decidir con cuál rellenar:

In [ ]:
print("Media:", df['Arrival Delay in Minutes'].mean())
print("Mediana:", df['Arrival Delay in Minutes'].median())
print("% de vuelos sin demora de llegada:", (df['Arrival Delay in Minutes'] == 0).mean() * 100)

La mediana es 0 y más de la mitad de los vuelos (56.3%) no tiene demora. La media (15) está inflada por los pocos vuelos con demoras muy largas. Por eso rellenamos con la mediana y no con la media, así no metemos un valor que no representa a la mayoría.

In [ ]:
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())
print("Faltantes restantes:", df.isnull().sum().sum())

### Faltantes ocultos: los ceros en las calificaciones

El `isnull()` solo encuentra los NaN, pero hay otro tipo de faltante que no aparece ahí. Las 14 preguntas de la encuesta se califican de 1 a 5, y según la descripción del dataset el **0 significa "no aplica / no respondió"**. O sea que un 0 no es una nota baja, es una respuesta que no existe.

Contamos cuántos ceros tiene cada pregunta:

In [ ]:
calificaciones = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
                  'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
                  'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling',
                  'Checkin service', 'Inflight service', 'Cleanliness']

ceros = (df[calificaciones] == 0).sum()
ceros_pct = (ceros / len(df) * 100).round(2)
resumen_ceros = pd.DataFrame({'ceros': ceros, 'porcentaje': ceros_pct}).sort_values('ceros', ascending=False)
print(resumen_ceros)
print("\nFilas con al menos un 0:", (df[calificaciones] == 0).any(axis=1).sum())

Hay más faltantes ocultos (10,313 filas con al menos un 0) que faltantes "normales" (393). Se concentran en cuatro preguntas: `Departure/Arrival time convenient` (5.1%), `Ease of Online booking` (4.4%), `Inflight wifi service` (3.0%) y `Online boarding` (2.4%). En el resto son casos aislados.

Antes de decidir qué hacer con ellos, revisamos si los pasajeros que respondieron 0 se parecen al resto o no. Comparamos el % de satisfechos entre quienes pusieron 0 y el total:

In [ ]:
print(f"% satisfechos en todo el dataset: {(df['satisfaction'] == 'satisfied').mean() * 100:.1f}%\n")
for col in resumen_ceros[resumen_ceros['ceros'] > 100].index:
    pct_sat = (df.loc[df[col] == 0, 'satisfaction'] == 'satisfied').mean() * 100
    print(f"{col}: {pct_sat:.1f}% satisfechos entre los que respondieron 0")

Los ceros **no son aleatorios**. El caso más claro es el wifi: el 99.7% de quienes pusieron 0 están satisfechos, contra el 43.4% general. `Ease of Online booking` también se aleja bastante (66.6%). Probablemente son pasajeros que no usaron ese servicio (por ejemplo, vuelos sin wifi), y ese grupo en particular termina más satisfecho.

**Decisión:** no los reemplazamos por la mediana ni los tratamos como una nota. Si imputamos, se pierde justamente esa información. Lo que proponemos es:
- Para el análisis de las calificaciones (medias, correlaciones), tratar el 0 como NaN, para que no baje los promedios como si fuera una nota muy mala.
- Para un modelo más adelante, crear una variable indicadora (`no_respondio_wifi`, etc.) en las 4 preguntas con más ceros, porque el hecho de no responder ya dice algo sobre la satisfacción.

Por ahora no modificamos `df`: dejamos los ceros como están y los tenemos en cuenta en cada análisis.

### Duplicados

Por último revisamos si hay filas repetidas, y si algún `id` de pasajero aparece más de una vez (podría pasar al unir train y test):

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("IDs repetidos:", df['id'].duplicated().sum())
print("Filas duplicadas sin contar el id:", df.drop(columns='id').duplicated().sum())

No hay duplicados de ningún tipo: ni filas completas, ni `id` repetidos, ni pasajeros con exactamente las mismas respuestas bajo otro `id`. Cada fila es un pasajero distinto y train y test no se solapan.

## 4. Outliers

Primero vemos qué columnas numéricas hay en total, para no analizar outliers "a ojo":

In [ ]:
todas_columnas_numericas = df.select_dtypes(include=np.number).columns.tolist()
print(todas_columnas_numericas)

In [ ]:
df[todas_columnas_numericas].describe()

Con el `describe()` de arriba separamos las columnas en tres grupos:

- `id`: es solo un número de pasajero, no sirve para análisis.
- Las columnas de calificación (wifi, comida, comodidad del asiento, etc.) van de 0 a 5. Son escalas de encuesta, no variables continuas — no tiene sentido buscarles "outliers" (un 5 no es un valor raro, es la nota máxima).
- Las que sí son continuas de verdad: `Age`, `Flight Distance`, `Departure Delay in Minutes` y `Arrival Delay in Minutes`. En estas sí vale la pena buscar atípicos.

Nos quedamos con este último grupo para el análisis de outliers.

In [ ]:
columnas_numericas = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), columnas_numericas):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

`Age` no tiene outliers. `Flight Distance` tiene algunos, moderados. Los dos delays tienen muchísimos puntos fuera de la caja — algo esperado, ya que la mayoría de vuelos no tiene demora pero hay una cola de vuelos con demoras grandes.

Ahora lo calculamos con el método IQR para tener un número exacto, no solo la lectura visual:

In [ ]:
def detectar_outliers_iqr(serie):
    Q1, Q3 = serie.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = (serie < lim_inf) | (serie > lim_sup)
    return outliers.sum(), lim_inf, lim_sup

for col in columnas_numericas:
    n_outliers, li, ls = detectar_outliers_iqr(df[col])
    pct = (n_outliers / len(df)) * 100
    print(f"{col}: {n_outliers} outliers ({pct:.2f}%) — rango válido [{li:.1f}, {ls:.1f}]")

Los delays salen con 13-14% de "outliers", un porcentaje demasiado alto para tratarlos como errores. Además, el límite que calcula el IQR (30 minutos) es poco realista para vuelos: una demora de una hora pasa todo el tiempo en la vida real, no es un dato dañado.

Por eso no eliminamos estos outliers. Son vuelos con demoras reales, y de hecho son justo el tipo de caso que puede interesar analizar más adelante (¿la demora afecta la satisfacción?).

### Comparación con z-score

El IQR no es el único método. El z-score mide a cuántas desviaciones estándar de la media está cada valor, y se suele marcar como atípico lo que tenga |z| > 3. Lo calculamos en las mismas 4 columnas para comparar:

In [ ]:
comparacion = []
for col in columnas_numericas:
    z = np.abs(stats.zscore(df[col]))
    n_iqr, _, _ = detectar_outliers_iqr(df[col])
    comparacion.append({
        'columna': col,
        'outliers_IQR': n_iqr,
        'outliers_z>3': (z > 3).sum(),
        '%_IQR': round(n_iqr / len(df) * 100, 2),
        '%_z>3': round((z > 3).sum() / len(df) * 100, 2),
        'limite_z (media + 3*std)': round(df[col].mean() + 3 * df[col].std(), 1),
    })
pd.DataFrame(comparacion).set_index('columna')

Los dos métodos dan resultados muy distintos:

- **Delays:** el IQR marca ~14% y el z-score solo ~2.1%. El IQR pone el límite en 30 minutos porque la mitad de los vuelos tiene demora 0 y la caja queda muy estrecha. El z-score lo pone en ~130 minutos (2 horas), un criterio más razonable para una demora "fuera de lo normal".
- **Flight Distance:** 2,855 por IQR contra 78 por z-score. Solo los vuelos de más de ~4,180 millas quedan fuera con z.
- **Age:** el IQR no marca ninguno y el z-score marca 25 (pasajeros de 85 años). Son edades posibles, no errores.

Ninguno de los dos métodos es perfecto aquí: el z-score asume una distribución más o menos normal y usa la media y la desviación estándar, que también se inflan con la cola. Pero los dos llevan a la misma conclusión: los valores extremos son pocos, son posibles en la vida real y no hay por qué eliminarlos.

### Diagramas de dispersión

Para ver los outliers en dos dimensiones y cómo se relacionan con la satisfacción, graficamos una muestra de 10,000 pasajeros (con el dataset completo los puntos se tapan entre sí):

In [ ]:
muestra = df.sample(10000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=muestra, x='Departure Delay in Minutes', y='Arrival Delay in Minutes',
                hue='satisfaction', alpha=0.4, s=15, ax=axes[0])
axes[0].set_title('Demora de salida vs demora de llegada')

sns.scatterplot(data=muestra, x='Flight Distance', y='Arrival Delay in Minutes',
                hue='satisfaction', alpha=0.4, s=15, ax=axes[1])
axes[1].set_title('Distancia del vuelo vs demora de llegada')
plt.tight_layout()
plt.show()

- **Salida vs llegada:** los puntos forman casi una línea recta de pendiente 1. El avión que sale tarde llega tarde en casi la misma cantidad de minutos (la diferencia tiene mediana 0). Por eso Pearson da 0.96: los outliers no rompen la relación, al contrario, están justo sobre la línea y la refuerzan. Las dos variables miden casi lo mismo, lo cual es importante para la Fase 3 (son redundantes).
- **Distancia vs llegada:** no hay ninguna forma, es una nube. La correlación es prácticamente 0 (Spearman ≈ 0.00), así que los vuelos largos no se demoran más.
- **Satisfacción:** en los dos gráficos los colores están mezclados, incluso en las demoras grandes. La demora sola no separa a satisfechos de insatisfechos. En cambio, en el segundo gráfico se ve que los satisfechos se concentran más a la derecha (vuelos largos): la mediana de distancia es 1,249 millas en satisfechos contra 674 en insatisfechos.

**Un detalle que muestra el gráfico:** hay algunos puntos sobre el eje horizontal, con mucha demora de salida pero 0 de llegada, que se salen de la línea. Los revisamos:

In [ ]:
imputados = df_train['Arrival Delay in Minutes'].isna().tolist() + df_test['Arrival Delay in Minutes'].isna().tolist()
imputados = pd.Series(imputados, index=df.index)

raros = (df['Departure Delay in Minutes'] > 60) & (df['Arrival Delay in Minutes'] == 0)
print("Vuelos con más de 60 min de demora de salida y 0 de llegada:", raros.sum())
print("De esos, cuántos eran faltantes imputados con la mediana:", (raros & imputados).sum())

Todos esos puntos raros son filas donde `Arrival Delay in Minutes` venía vacía y la rellenamos con la mediana (0). La mediana era mejor opción que la media, pero el gráfico muestra que había una mejor: como la demora de llegada es casi igual a la de salida (correlación 0.96), se podría haber rellenado cada faltante con su propia `Departure Delay in Minutes`. Son solo 393 filas (0.3%), así que no cambia las conclusiones, pero lo dejamos anotado como mejora para el preprocesamiento.

## 5. Distribución de las variables numéricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), columnas_numericas):
    sns.histplot(df[col], kde=True, ax=ax, bins=40)
    ax.set_title(col)
plt.tight_layout()
plt.show()

`Age` tiene dos picos (uno cerca de los 25 y otro cerca de los 40), pero es bastante simétrica en general. `Flight Distance` tiene cola hacia la derecha (más vuelos cortos que largos). Los delays son casi todo ceros con una cola larguísima.

Confirmamos esto con el coeficiente de asimetría (skewness):

In [ ]:
for col in columnas_numericas:
    print(f"{col}: skewness = {df[col].skew():.2f}")

`Age` sale casi en 0 (simétrica), `Flight Distance` en 1.11 (sesgo moderado), y los dos delays por encima de 6.5 — un sesgo extremo. Cualquier valor de skewness fuera de ±1 ya se considera sesgado, así que los delays están muy lejos de una distribución normal. Esto confirma por qué el IQR no funcionó bien con ellos: ese método asume datos más simétricos.

Si más adelante se necesitara usar estas variables en un modelo, se podría aplicar una transformación logarítmica para suavizar ese sesgo.

Lo comprobamos: comparamos los histogramas de los delays antes y después de aplicar `np.log1p` (log(1 + x), que funciona con los ceros porque log1p(0) = 0):

In [ ]:
delays = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for fila, col in enumerate(delays):
    sns.histplot(df[col], bins=50, ax=axes[fila, 0])
    axes[fila, 0].set_title(f"{col} (original)")
    sns.histplot(np.log1p(df[col]), bins=50, ax=axes[fila, 1], color='darkorange')
    axes[fila, 1].set_title(f"log1p({col})")
plt.tight_layout()
plt.show()

for col in delays:
    print(f"{col}: skewness original = {df[col].skew():.2f} | con log1p = {np.log1p(df[col]).skew():.2f}")

Con `log1p` la asimetría baja de ~6.8 a menos de 1, y la cola larga se comprime: una demora de 1,500 minutos ya no queda a cientos de unidades del resto. No queda una campana perfecta, porque el 56% de los vuelos tiene demora 0 y eso forma una barra muy alta en el cero que ninguna transformación puede repartir. Aun así, es suficiente para que los delays no dominen métodos sensibles a la escala como el PCA. Por eso en la Fase 3 se aplica `log1p` antes de escalar.

## 6. Variables categóricas

In [ ]:
columnas_categoricas = df.select_dtypes(include=['object', 'str']).columns.tolist()
print(columnas_categoricas)

In [ ]:
for col in columnas_categoricas:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(normalize=True).round(3) * 100)

- `Gender`: casi mitad y mitad.
- `Customer Type`: 82% son clientes leales, solo 18% no leales.
- `Type of Travel`: 69% viaja por trabajo, 31% por motivos personales.
- `Class`: Business y Eco están parejos (48% y 45%), Eco Plus es minoría (7%).
- `satisfaction` (la variable que queremos explicar): 57% no satisfechos, 43% satisfechos. No es un desbalance grave.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), columnas_categoricas):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.show()

## 7. Relación entre variables

### Correlaciones numéricas

Antes de calcular correlaciones, comparamos Pearson contra Spearman en los delays, ya que sabemos que tienen un sesgo fuerte:

In [ ]:
for col1, col2 in [('Departure Delay in Minutes', 'Arrival Delay in Minutes'),
                     ('Departure Delay in Minutes', 'Age'),
                     ('Arrival Delay in Minutes', 'Flight Distance')]:
    pearson = df[col1].corr(df[col2])
    spearman, _ = spearmanr(df[col1], df[col2])
    print(f"{col1} vs {col2}: Pearson={pearson:.3f}, Spearman={spearman:.3f}")

Entre los dos delays, Pearson da 0.96 y Spearman da 0.74 — una diferencia real. Pearson se deja influenciar por los valores extremos y exagera un poco la relación; Spearman, al trabajar con rangos, da un número más realista. En las otras dos comparaciones, donde no hay relación real, da casi lo mismo con cualquiera de los dos métodos.

Con esto en mente, vemos la matriz de correlación completa (incluyendo algunas columnas de calificación):

In [ ]:
columnas_correlacion = columnas_numericas + ['Inflight wifi service', 'Online boarding', 'Seat comfort', 'Inflight entertainment']

corr = df[columnas_correlacion].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

Lo más fuerte es la relación entre los dos delays (0.96, ya la vimos) y entre `Seat comfort` e `Inflight entertainment` (0.61). Los delays no se relacionan con casi nada más.

### Tablas cruzadas con satisfacción

Ahora cruzamos las categóricas contra `satisfaction`, para ver cuáles parecen influir de verdad:

In [ ]:
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class']:
    print(f"\n--- {col} vs satisfaction ---")
    tabla = pd.crosstab(df[col], df['satisfaction'], normalize='index') * 100
    print(tabla.round(1))

`Gender` no marca casi diferencia (57% vs 56%). Las otras tres sí:

- `Customer Type`: los leales están satisfechos el 48% de las veces, los no leales solo el 24%.
- `Type of Travel`: es la más marcada de todas — negocios 58% satisfechos, personal solo 10%.
- `Class`: Business 69% satisfechos, Eco 19%, Eco Plus 25%.

`Type of Travel` y `Class` se mueven muy parecido. Puede que no sean dos efectos separados, sino el mismo fenómeno visto dos veces — lo revisamos:

In [ ]:
pd.crosstab(df['Type of Travel'], df['Class'], normalize='index').round(3) * 100

Confirmado: 66% de los que viajan por negocios van en Business, y 82% de los que viajan por motivos personales van en Eco. Están fuertemente ligadas, así que su efecto sobre la satisfacción probablemente se solapa.

## 8. Hipótesis y validación estadística

A partir de los patrones anteriores, planteamos:

- **H1:** el tipo de viaje influye en la satisfacción (negocios más satisfechos que personal).
- **H2:** la clase influye en la satisfacción (Business más satisfecho que Eco/Eco Plus).
- **H3:** la lealtad del cliente influye en la satisfacción.
- **H4:** `Type of Travel` y `Class` están relacionadas entre sí, así que su efecto sobre la satisfacción puede estar solapado — habría que separarlo con un modelo más adelante.

Probamos H1 con una prueba chi-cuadrado:

H0: no hay relación entre tipo de viaje y satisfacción.
H1: sí hay relación.

In [ ]:
tabla_contingencia = pd.crosstab(df['Type of Travel'], df['satisfaction'])
print(tabla_contingencia)

chi2, p_valor, gl, esperado = chi2_contingency(tabla_contingencia)
print(f"\nChi-cuadrado: {chi2:.2f}")
print(f"Grados de libertad: {gl}")
print(f"Valor p: {p_valor}")

El valor p sale prácticamente 0, mucho menor que 0.05, así que rechazamos H0: sí hay relación entre tipo de viaje y satisfacción, y no es casualidad.

Vale la pena aclarar algo: con una muestra tan grande (129,880 filas), el chi-cuadrado casi siempre va a salir significativo, incluso con diferencias chiquitas. Aquí no es el caso — la diferencia real (58% vs 10%) es enorme, así que el resultado importa tanto estadística como prácticamente.

## 9. Conclusiones

- El tipo de viaje es el factor que más se relaciona con la satisfacción (58% vs 10%), y lo confirmamos con la prueba chi-cuadrado.
- La clase de viaje está pegada al tipo de viaje (negocios → Business, personal → Eco), así que probablemente no son dos efectos independientes sino el mismo patrón visto de dos formas.
- La lealtad del cliente también influye, aunque menos que las dos anteriores.
- Los delays tienen una distribución muy sesgada (skewness ~6.8). El IQR marcó 14% como "outliers", pero decidimos no quitarlos porque son demoras reales, no errores.
- Pearson exagera un poco la correlación entre los delays por su sesgo; Spearman da un número más confiable (0.96 vs 0.74).
- El género no parece influir en la satisfacción.

**Problemas de calidad encontrados:**
- Columna `Unnamed: 0` sin utilidad (se eliminó).
- 393 faltantes en `Arrival Delay in Minutes` (0.3%), rellenados con la mediana.

**Para seguir más adelante:**
- Separar el efecto de `Type of Travel` y `Class` con un modelo de regresión o clasificación, ya que están relacionadas entre sí.
- Probar H2 y H3 también con chi-cuadrado.
- Ver si la duración de la demora (no solo si hay o no) afecta la satisfacción.